In [2]:
from lecture import conversion
import pandas as pd

df_rera = pd.read_parquet("../data/rer_a_2026-11-24.parquet")
df_rerb = pd.read_parquet("../data/rer_b_2026-11-24.parquet")
df_rerd = pd.read_parquet("../data/rer_d_2026-11-24.parquet")
df_incidents = pd.read_parquet("../data/incidents_2026-11-24.parquet")

In [3]:
df_rera = df_rera[df_rera["date"].between("2026-10-02", "2026-11-23")]
df_rerb = df_rerb[df_rerb["date"].between("2026-10-02", "2026-11-23")]
df_rerd = df_rerd[df_rerd["date"].between("2026-10-02", "2026-11-23")]
df_incidents = df_incidents[df_incidents["date"].between("2026-10-02", "2026-11-23")]

df_rera = conversion(df_rera)
df_rerb = conversion(df_rerb)
df_rerd = conversion(df_rerd)

In [4]:
df_rera.columns

Index(['journey_id', 'line', 'mission', 'origin', 'destination', 'direction',
       'is_long', 'nb_stops', 'total_delay_sec', 'max_delay_added_sec',
       'worst_stop', 'disappeared', 'first_seen', 'last_seen', 'date',
       'stop_name', 'aimed_time', 'expected_time', 'delay_sec', 'platform',
       'status', 'stop_order', 'delay_added_sec'],
      dtype='str')

In [5]:
# stations centrales sur chaque ligne

gares_a = [
    "Charles De Gaulle-Étoile",
    "Auber",
    "Châtelet-Les Halles",
    "Gare de Lyon",
    "Nation",
]

gares_b = [
    "Denfert-Rochereau",
    "Port Royal",
    "Luxembourg",
    "Saint-Michel Notre-Dame",
    "Châtelet-Les Halles",
    "Gare du Nord",
    "La Plaine Stade de France",
    "La Courneuve - Aubervilliers",
    "Le Bourget",
    "Drancy",
    "Le Blanc-Mesnil",
    "Aulnay-sous-Bois",
]

gares_d = [
    "Gare de Lyon",
    "Châtelet-Les Halles",
    "Gare du Nord",
    "Stade de France-Saint-Denis",
    "Saint-Denis",
]

# uniformisation d'une variante sur le b
df_rerb["stop_name"] = df_rerb["stop_name"].replace({"Châtelet - Les Halles": "Châtelet-Les Halles"})

df_rera = df_rera[df_rera["stop_name"].isin(gares_a)].copy()
df_rerb = df_rerb[df_rerb["stop_name"].isin(gares_b)].copy()
df_rerd = df_rerd[df_rerd["stop_name"].isin(gares_d)].copy()

In [ ]:
# retard, suppressions, tunnel et créneaux horaires

def ajout_variables(df):
    df = df.copy()
    df["retard_min"] = df["delay_sec"] / 60
    df["est_supprime"] = df["status"] == "CANCELLED"
    df["zone_tunnel"] = df["stop_name"].isin(["Châtelet-Les Halles", "Gare du Nord"])

    semaine = df["aimed_time"].dt.day_of_week < 5
    heure = df["aimed_time"].dt.hour + df["aimed_time"].dt.minute / 60
    pointe_matin = semaine & (heure >= 7.5) & (heure < 9.5)
    pointe_soir = semaine & (heure >= 17.0) & (heure < 19.5)

    df["heure_pointe"] = pointe_matin | pointe_soir

    tranche = pd.Series("Heures creuses", index=df.index)
    tranche.loc[~semaine] = "Week-end"
    tranche.loc[semaine & ((heure >= 20.0) | (heure < 6.0))] = "Soirée"
    tranche.loc[pointe_matin] = "Pointe matin"
    tranche.loc[pointe_soir] = "Pointe soir"
    df["tranche_horaire"] = tranche
    return df

df_rera = ajout_variables(df_rera)
df_rerb = ajout_variables(df_rerb)
df_rerd = ajout_variables(df_rerd)

In [7]:
# controle et retards aberrants

for nom, df in [("RER A", df_rera), ("RER B", df_rerb), ("RER D", df_rerd)]:
    aberrants = (df["retard_min"].abs() > 180).sum()
    print(f"=== {nom} ({len(df)} arrêts) ===")
    print(f"  Suppressions : {df['est_supprime'].mean() * 100:.2f} % | Tunnel : {df['zone_tunnel'].mean() * 100:.1f} % | Pointe : {df['heure_pointe'].mean() * 100:.1f} %")
    print(f"  Retard (min) médian={df['retard_min'].median():.2f}, P90={df['retard_min'].quantile(0.9):.2f}, aberrants (|r|>180 min)={aberrants}")
    print("  Tranches :", df["tranche_horaire"].value_counts().to_dict())

=== RER A (148666 arrêts) ===
  Suppressions : 0.81 % | Tunnel : 20.2 % | Pointe : 27.2 %
  Retard (min) médian=0.37, P90=4.32, aberrants (|r|>180 min)=5
  Tranches : {'Heures creuses': 54935, 'Week-end': 32915, 'Pointe soir': 21442, 'Soirée': 20353, 'Pointe matin': 19021}
=== RER B (266064 arrêts) ===
  Suppressions : 2.00 % | Tunnel : 18.3 % | Pointe : 26.0 %
  Retard (min) médian=1.43, P90=8.47, aberrants (|r|>180 min)=1636
  Tranches : {'Heures creuses': 95537, 'Week-end': 61799, 'Soirée': 39571, 'Pointe soir': 37781, 'Pointe matin': 31376}
=== RER D (60314 arrêts) ===
  Suppressions : 4.21 % | Tunnel : 38.7 % | Pointe : 26.4 %
  Retard (min) médian=0.87, P90=7.75, aberrants (|r|>180 min)=1
  Tranches : {'Heures creuses': 25232, 'Week-end': 11104, 'Pointe soir': 9115, 'Soirée': 8074, 'Pointe matin': 6789}
